
# Clean MLP overfitting sweep notebook

This notebook keeps only the components needed to answer:

**For each permutation length `n`, at what MLP width does the model start to overfit?**

It does four things:
1. generate random-walk training/test data,
2. train an MLP for each `(n, width)` pair,
3. compare train vs. test metrics,
4. summarize the overfitting threshold for each `n`.


In [1]:
print("hello world")

hello world


In [2]:

import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import r2_score, root_mean_squared_error
from scipy import stats

# -----------------------------
# Base config
# -----------------------------
CFG = {}
CFG['random_walks_type'] = 'non-backtracking-beam'   # 'simple' or 'non-backtracking-beam'
CFG['n_random_walks_to_generate'] = 10_000           # training samples generated each epoch
CFG['n_random_walks_steps_back_to_ban'] = 8          # only used for non-backtracking-beam
CFG['n_epochs'] = 10
CFG['batch_size'] = 1024
CFG['lr'] = 0.001
CFG['model_type'] = 'MLP'

# Sweep values
n_values = [8, 12, 16, 20, 24, 28]
widths = [4, 8, 16, 24, 32, 48, 64, 128, 256, 512, 1024]

# Evaluation settings
N_TEST_SAMPLES = 1024
RESULTS_CSV = 'mlp_width_n_overfit_sweep_clean.csv'
SUMMARY_CSV = 'mlp_width_n_overfit_summary_clean.csv'

# Reproducibility
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cuda


In [3]:

# -----------------------------
# Problem / random-walk helpers
# -----------------------------
def get_LRX_moves(n):
    L = np.array(list(np.arange(1, n)) + [0])
    R = np.array([n - 1] + list(np.arange(n - 1)))
    X = np.array([1, 0] + list(np.arange(2, n)))
    return L, R, X


def build_problem(n, device=device):
    L, R, X = get_LRX_moves(n)
    list_generators = [L, R, X]
    dtype_generators = torch.int64
    state_destination = torch.arange(n, device=device, dtype=dtype_generators)
    dtype_state = torch.uint8 if n <= 256 else torch.uint16
    return list_generators, state_destination, dtype_state


def get_neighbors(states, moves):
    return torch.gather(
        states.unsqueeze(1).expand(states.size(0), moves.shape[0], states.size(1)),
        2,
        moves.unsqueeze(0).expand(states.size(0), moves.shape[0], states.size(1))
    )


def _normalize_generators(generators, device):
    if isinstance(generators, list):
        list_generators = generators
    elif isinstance(generators, tuple):
        list_generators = list(generators)
    elif isinstance(generators, torch.Tensor):
        list_generators = [list(generators[i, :]) for i in range(generators.shape[0])]
    elif isinstance(generators, np.ndarray):
        list_generators = [list(generators[i, :]) for i in range(generators.shape[0])]
    else:
        raise ValueError(f'Unsupported format for generators: {type(generators)}')
    tensor_generators = torch.tensor(list_generators, device=device, dtype=torch.int64)
    return list_generators, tensor_generators


def _normalize_start_state(state_rw_start, state_size, n_random_walks_to_generate, device, dtype):
    if (state_rw_start == '01234...') or (state_rw_start == 'Auto'):
        state_rw_start = torch.arange(state_size, device=device, dtype=dtype).reshape(-1, state_size)
    elif isinstance(state_rw_start, torch.Tensor):
        state_rw_start = state_rw_start.to(device).to(dtype).reshape(-1, state_size)
    else:
        state_rw_start = torch.tensor(state_rw_start, device=device, dtype=dtype).reshape(-1, state_size)
    return state_rw_start.view(1, state_size).expand(n_random_walks_to_generate, state_size).clone()


def random_walks_simple(
    generators,
    n_random_walk_length,
    n_random_walks_to_generate,
    state_rw_start='01234...',
    device=device,
    dtype='Auto',
    verbose=0,
):
    list_generators, tensor_generators = _normalize_generators(generators, device)
    state_size = len(list_generators[0])
    n_generators = len(list_generators)

    if dtype == 'Auto':
        dtype = torch.uint8 if state_size <= 256 else torch.uint16

    array_of_states = _normalize_start_state(
        state_rw_start, state_size, n_random_walks_to_generate, device, dtype
    )

    X = torch.zeros(
        n_random_walks_to_generate * n_random_walk_length,
        state_size,
        device=device,
        dtype=dtype,
    )
    y = torch.zeros(
        n_random_walks_to_generate * n_random_walk_length,
        device=device,
        dtype=torch.uint32,
    )

    X[:n_random_walks_to_generate, :] = array_of_states
    y[:n_random_walks_to_generate] = 0

    row_indices = np.arange(array_of_states.shape[0])[:, np.newaxis]

    for i_step in range(1, n_random_walk_length):
        y[i_step * n_random_walks_to_generate : (i_step + 1) * n_random_walks_to_generate] = i_step
        IX_moves = np.random.randint(0, n_generators, size=n_random_walks_to_generate, dtype=int)
        new_array_of_states = array_of_states[row_indices, tensor_generators[IX_moves, :]]
        array_of_states = new_array_of_states
        X[i_step * n_random_walks_to_generate : (i_step + 1) * n_random_walks_to_generate, :] = new_array_of_states

    return X, y


def random_walks_nbt(
    generators,
    n_random_walk_length,
    n_random_walks_to_generate,
    state_rw_start='01234...',
    n_random_walks_steps_back_to_ban=0,
    device=device,
    dtype='Auto',
    vec_hasher='Auto',
    verbose=0,
):
    list_generators, tensor_generators = _normalize_generators(generators, device)
    state_size = len(list_generators[0])
    n_generators = len(list_generators)

    if dtype == 'Auto':
        dtype = torch.uint8 if state_size <= 256 else torch.uint16

    array_current_states = _normalize_start_state(
        state_rw_start, state_size, n_random_walks_to_generate, device, dtype
    )

    if vec_hasher == 'Auto':
        max_int = int(2**62)
        vec_hasher = torch.randint(
            -max_int, max_int + 1, size=(state_size,), device=device, dtype=torch.int64
        )

    X = torch.zeros(
        n_random_walks_to_generate * n_random_walk_length,
        state_size,
        device=device,
        dtype=dtype,
    )
    y = torch.zeros(
        n_random_walks_to_generate * n_random_walk_length,
        device=device,
        dtype=torch.uint32,
    )

    X[:n_random_walks_to_generate, :] = array_current_states
    y[:n_random_walks_to_generate] = 0

    if n_random_walks_steps_back_to_ban > 0:
        hash_initial_state = torch.sum(array_current_states[:1] * vec_hasher, dim=1)
        vec_hashes_current = hash_initial_state.expand(
            n_random_walks_to_generate * n_generators,
            n_random_walks_steps_back_to_ban,
        ).clone()
        i_cyclic_index_for_hash_storage = 0

    i_step_corrected = 0
    for i_step in range(1, n_random_walk_length):
        array_new_states = get_neighbors(array_current_states, tensor_generators).flatten(end_dim=1)
        vec_hashes_new = torch.sum(array_new_states * vec_hasher, dim=1)

        if n_random_walks_steps_back_to_ban > 0:
            mask_new = ~torch.isin(vec_hashes_new, vec_hashes_current.view(-1), assume_unique=False)
            mask_new_sum = mask_new.sum().item()
            if mask_new_sum >= n_random_walks_to_generate:
                array_new_states = array_new_states[mask_new, :]
                i_step_corrected += 1
            elif mask_new_sum > 0:
                i_tmp0 = int(np.ceil(n_random_walks_to_generate / mask_new_sum))
                array_new_states = array_new_states[mask_new, :].repeat(i_tmp0, 1)[:n_random_walks_to_generate, :]
                i_step_corrected += 1
            else:
                array_new_states = array_current_states

        perm = torch.randperm(array_new_states.size(0), device=device)
        array_current_states = array_new_states[perm][:n_random_walks_to_generate]

        y[i_step * n_random_walks_to_generate : (i_step + 1) * n_random_walks_to_generate] = i_step_corrected
        X[i_step * n_random_walks_to_generate : (i_step + 1) * n_random_walks_to_generate, :] = array_current_states

        if n_random_walks_steps_back_to_ban > 0:
            i_cyclic_index_for_hash_storage = (i_cyclic_index_for_hash_storage + 1) % n_random_walks_steps_back_to_ban
            vec_hashes_current[:, i_cyclic_index_for_hash_storage] = vec_hashes_new

    return X, y


def random_walks(
    generators,
    n_random_walk_length,
    n_random_walks_to_generate,
    state_rw_start='01234...',
    n_random_walks_steps_back_to_ban=0,
    random_walks_type='simple',
    device=device,
    dtype='Auto',
    vec_hasher='Auto',
    verbose=0,
):
    if random_walks_type == 'non-backtracking-beam':
        return random_walks_nbt(
            generators=generators,
            n_random_walk_length=n_random_walk_length,
            n_random_walks_to_generate=n_random_walks_to_generate,
            state_rw_start=state_rw_start,
            n_random_walks_steps_back_to_ban=n_random_walks_steps_back_to_ban,
            device=device,
            dtype=dtype,
            vec_hasher=vec_hasher,
            verbose=verbose,
        )
    return random_walks_simple(
        generators=generators,
        n_random_walk_length=n_random_walk_length,
        n_random_walks_to_generate=n_random_walks_to_generate,
        state_rw_start=state_rw_start,
        device=device,
        dtype=dtype,
        verbose=verbose,
    )


In [4]:

# -----------------------------
# Model and metric helpers
# -----------------------------
class Net(nn.Module):
    def __init__(self, input_size, hidden_dims, num_classes_for_one_hot):
        super().__init__()
        self.num_classes_for_one_hot = num_classes_for_one_hot
        in_features = input_size * num_classes_for_one_hot
        layers = []
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(in_features, hidden_dim))
            layers.append(nn.ReLU())
            in_features = hidden_dim
        layers.append(nn.Linear(in_features, 1))
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        x = torch.nn.functional.one_hot(
            x.long(), num_classes=self.num_classes_for_one_hot
        ).float().flatten(start_dim=-2)
        return self.layers(x)


def get_model_inf(CFG):
    s = str(CFG['model_type'])
    for t in CFG['list_layers_sizes']:
        s += f'_{t}'
    s += f"_epochs{CFG['n_epochs']}"
    s += f"_rwlen{CFG['n_random_walk_length']}"
    s += f"_n_rw{CFG['n_random_walks_to_generate']}"
    return s


def evaluate_predictions(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'rmse': root_mean_squared_error(y_true, y_pred),
        'spearman': stats.spearmanr(y_true, y_pred).statistic,
    }


In [7]:

# -----------------------------
# Sweep runner
# -----------------------------
def run_overfit_sweep(n_values, widths, cfg=CFG, n_test_samples=N_TEST_SAMPLES):
    results = []

    random_walks_type = cfg['random_walks_type']
    n_random_walks_to_generate = cfg['n_random_walks_to_generate']
    n_random_walks_steps_back_to_ban = cfg['n_random_walks_steps_back_to_ban']
    n_epochs = cfg['n_epochs']
    batch_size = cfg['batch_size']
    lr = cfg['lr']

    for n in n_values:
        print(f'\n==============================')
        print(f'SWEEPING n = {n}')
        print(f'==============================')

        cfg['n_permutations_length'] = n
        cfg['n_random_walk_length'] = int(n * (n - 1) / 2)
        cfg['input_size'] = n

        list_generators, state_destination, dtype_state = build_problem(n)

        X_test, y_test = random_walks(
            list_generators,
            n_random_walk_length=cfg['n_random_walk_length'],
            n_random_walks_to_generate=n_test_samples,
            n_random_walks_steps_back_to_ban=n_random_walks_steps_back_to_ban,
            random_walks_type=random_walks_type,
            state_rw_start=state_destination,
            dtype=dtype_state,
        )
        test_dataset = TensorDataset(X_test)
        test_loader = DataLoader(test_dataset, batch_size=batch_size)

        for width in widths:
            print(f'\n----- n={n}, width={width} -----')
            cfg['list_layers_sizes'] = [width]
            str_modeling_inf = get_model_inf(cfg)

            model = Net(
                input_size=n,
                hidden_dims=cfg['list_layers_sizes'],
                num_classes_for_one_hot=n,
            ).to(device)

            criterion = nn.MSELoss()
            optimizer = optim.Adam(model.parameters(), lr=lr)

            t_train_start = time.time()
            list_epoch_train_loss = []
            X_train_last = None
            y_train_last = None

            for epoch in range(n_epochs):
                X_train, y_train = random_walks(
                    list_generators,
                    n_random_walk_length=cfg['n_random_walk_length'],
                    n_random_walks_to_generate=n_random_walks_to_generate,
                    n_random_walks_steps_back_to_ban=n_random_walks_steps_back_to_ban,
                    random_walks_type=random_walks_type,
                    state_rw_start=state_destination,
                    dtype=dtype_state,
                )

                y_train = y_train.float()
                indices = torch.randperm(X_train.shape[0], device=X_train.device)
                X_train = X_train[indices]
                y_train = y_train[indices]

                X_train_last = X_train
                y_train_last = y_train

                model.train()
                n_states_all = X_train.shape[0]
                train_loss = 0.0
                cc = 0

                for i_start_batch in range(0, n_states_all, batch_size):
                    i_end_batch = min(i_start_batch + batch_size, n_states_all)
                    batch_X = X_train[i_start_batch:i_end_batch]
                    batch_y = y_train[i_start_batch:i_end_batch]

                    outputs = model(batch_X).squeeze()
                    loss = criterion(outputs, batch_y)

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    train_loss += loss.item()
                    cc += 1

                train_loss /= cc
                list_epoch_train_loss.append(train_loss)
                print(f'epoch {epoch}: train_loss={train_loss:.4f}')

            fit_time = time.time() - t_train_start

            model.eval()
            train_dataset = TensorDataset(X_train_last)
            train_loader = DataLoader(train_dataset, batch_size=batch_size)
            
            train_pred_list = []
            
            model.eval()
            # with torch.no_grad():
            #     train_pred = model(X_train_last).detach().cpu().numpy().ravel()
            with torch.no_grad():
                for batch in train_loader:
                    batch_X = batch[0].to(device)
                    batch_pred = model(batch_X)
                    train_pred_list.append(batch_pred.detach().cpu())
            
            train_pred = torch.cat(train_pred_list).numpy().ravel()

            y_train_true = y_train_last.detach().cpu().numpy().ravel()
            train_metrics = evaluate_predictions(y_train_true, train_pred)

            y_pred_list = []
            t_pred_start = time.time()
            with torch.no_grad():
                for batch in test_loader:
                    batch_X = batch[0].to(device)
                    batch_pred = model(batch_X)
                    y_pred_list.append(batch_pred.detach().cpu())
            predict_time = time.time() - t_pred_start

            y_pred = torch.cat(y_pred_list).numpy().ravel()
            y_true = y_test.detach().cpu().numpy().ravel()
            test_metrics = evaluate_predictions(y_true, y_pred)

            results.append({
                'n_permutations_length': n,
                'width': width,
                'layers': str(cfg['list_layers_sizes']),
                'model': str_modeling_inf,
                'final_train_loss': list_epoch_train_loss[-1],
                'train_r2': train_metrics['r2'],
                'train_rmse': train_metrics['rmse'],
                'train_spearman': train_metrics['spearman'],
                'test_r2': test_metrics['r2'],
                'test_rmse': test_metrics['rmse'],
                'test_spearman': test_metrics['spearman'],
                'rmse_gap': test_metrics['rmse'] - train_metrics['rmse'],
                'spearman_gap': train_metrics['spearman'] - test_metrics['spearman'],
                'fit_time_sec': fit_time,
                'predict_time_sec': predict_time,
                'n_epochs': n_epochs,
                'n_random_walks_to_generate': n_random_walks_to_generate,
                'n_random_walk_length': cfg['n_random_walk_length'],
            })

    df = pd.DataFrame(results).sort_values(['n_permutations_length', 'width']).reset_index(drop=True)
    return df


In [ ]:

# Run the full sweep
# This can take a while. For a quick smoke test, temporarily shrink n_values/widths or set CFG['n_epochs']=1.

df_combo_sweep = run_overfit_sweep(n_values=n_values, widths=widths, cfg=CFG)
display(df_combo_sweep)

df_combo_sweep.to_csv(RESULTS_CSV, index=False)
print(f'Saved detailed results to {RESULTS_CSV}')



SWEEPING n = 8

----- n=8, width=4 -----
epoch 0: train_loss=204.2575
epoch 1: train_loss=100.0943
epoch 2: train_loss=63.8677
epoch 3: train_loss=59.9293
epoch 4: train_loss=58.5052
epoch 5: train_loss=58.6767
epoch 6: train_loss=58.4846
epoch 7: train_loss=57.2920
epoch 8: train_loss=55.0938
epoch 9: train_loss=59.9538

----- n=8, width=8 -----
epoch 0: train_loss=167.7868
epoch 1: train_loss=64.7229
epoch 2: train_loss=48.9671
epoch 3: train_loss=32.4072
epoch 4: train_loss=17.6801
epoch 5: train_loss=8.9032
epoch 6: train_loss=6.2609
epoch 7: train_loss=4.9041
epoch 8: train_loss=3.1792
epoch 9: train_loss=3.1819

----- n=8, width=16 -----
epoch 0: train_loss=147.7161
epoch 1: train_loss=57.0407
epoch 2: train_loss=39.5980
epoch 3: train_loss=16.1454
epoch 4: train_loss=5.9405
epoch 5: train_loss=2.6080
epoch 6: train_loss=2.4910
epoch 7: train_loss=2.0259
epoch 8: train_loss=3.0847
epoch 9: train_loss=2.0928

----- n=8, width=24 -----
epoch 0: train_loss=147.2449
epoch 1: train_l

In [ ]:

# -----------------------------
# Overfitting summary
# -----------------------------
def summarize_overfitting(df):
    summary_rows = []

    for n in sorted(df['n_permutations_length'].unique()):
        df_n = df[df['n_permutations_length'] == n].sort_values('width').reset_index(drop=True)

        best_test_rmse = float('inf')
        overfit_width_rmse = None
        best_test_spearman = -float('inf')
        overfit_width_spearman = None

        for _, row in df_n.iterrows():
            if row['test_rmse'] < best_test_rmse:
                best_test_rmse = row['test_rmse']
            elif overfit_width_rmse is None:
                overfit_width_rmse = row['width']

            if row['test_spearman'] > best_test_spearman:
                best_test_spearman = row['test_spearman']
            elif overfit_width_spearman is None:
                overfit_width_spearman = row['width']

        best_rmse_row = df_n.loc[df_n['test_rmse'].idxmin()]
        best_spear_row = df_n.loc[df_n['test_spearman'].idxmax()]

        summary_rows.append({
            'n_permutations_length': n,
            'best_width_by_rmse': best_rmse_row['width'],
            'best_test_rmse': best_rmse_row['test_rmse'],
            'overfit_starts_by_rmse': overfit_width_rmse,
            'best_width_by_spearman': best_spear_row['width'],
            'best_test_spearman': best_spear_row['test_spearman'],
            'overfit_starts_by_spearman': overfit_width_spearman,
        })

    return pd.DataFrame(summary_rows)


df_overfit_summary = summarize_overfitting(df_combo_sweep)
display(df_overfit_summary)

df_overfit_summary.to_csv(SUMMARY_CSV, index=False)
print(f'Saved summary to {SUMMARY_CSV}')


In [ ]:

# -----------------------------
# Plots
# -----------------------------
for n in sorted(df_combo_sweep['n_permutations_length'].unique()):
    df_n = df_combo_sweep[df_combo_sweep['n_permutations_length'] == n].sort_values('width')

    plt.figure(figsize=(8, 5))
    plt.plot(df_n['width'], df_n['train_rmse'], marker='o', label='train RMSE')
    plt.plot(df_n['width'], df_n['test_rmse'], marker='o', label='test RMSE')
    plt.xscale('log', base=2)
    plt.title(f'RMSE vs width (n={n})')
    plt.xlabel('width')
    plt.ylabel('RMSE')
    plt.grid(True)
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(df_n['width'], df_n['train_spearman'], marker='o', label='train Spearman')
    plt.plot(df_n['width'], df_n['test_spearman'], marker='o', label='test Spearman')
    plt.xscale('log', base=2)
    plt.title(f'Spearman vs width (n={n})')
    plt.xlabel('width')
    plt.ylabel('Spearman')
    plt.grid(True)
    plt.legend()
    plt.show()



## Notes

- Overfitting starts where **training keeps improving but test performance starts getting worse**.
- `test_rmse` is lower-is-better.
- `test_spearman` is higher-is-better.
- Because training data is regenerated each epoch, the exact threshold may move a bit from run to run. For a more stable estimate, repeat the sweep 2–3 times and average.
- For a faster pilot run, reduce `n_values`, `widths`, `CFG['n_epochs']`, or `CFG['n_random_walks_to_generate']`.
